Section 1: 
Task type: Classification (binary)

Ranking Signal Analysis starts as exploratory by using correlations and group 
comparisons across signals. To make it concrete and testable, I'm framing 
it as binary classification: 
- Given a page's observable signals to make sure it in 
the declining group or not?. This turns the pattern I found in Week 1 
into something I can actually validate, 
rather than leaving it as a descriptive observation.

Section 2: Target or Proxy

Target : `is_declining = (trend_direction == "down")`

This matches the starter pipeline's own proxy label. It's a current-window 
bucket, not a true future outcome so it's a proxy, not a guaranteed 
ground truth. I'm using it because it's directly available and gives me a 
concrete binary target to test signal associations against in Week 2 and a
stronger capstone version would eventually redefine this as a 
future-window outcome (e.g., decline over the next 30 days), which the 
lane guide recommends over the current-window version.


Section 3: Success Metric

Primary metric: ROC AUC 
Secondary metric: Precision

Accuracy alone isn't a good fit here since "down" is already the majority 
class (54.2% from Week 1) and a model could score high accuracy just by 
guessing "down" every time without learning anything useful.

ROC AUC measures how well the model ranks declining pages above non-declining 
ones regardless of class balance, which better reflects the real use 
case: a reviewer doesn't need every page correctly labeled, they need 
the most likely decliners ranked near the top of their queue.

Precision is a useful as secondary metric because it directly 
answers "of the top pages I'd flag for review, how many are actually 
right?" and it is matching the output in practice.


Section 4: Unit of analysis, as a real dataframe

One row = one content page (`content_id`), evaluated on its most recent 
90-day window of signals.

Below: the working dataframe after applying the standard filters 
(`impressions_90d > 0`, `content_age_days >= 90`), with the target 
column added.

In [1]:
import pandas as pd

df = pd.read_csv("/Users/anhvuong/flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv")
filtered = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
filtered["is_declining"] = (filtered["trend_direction"] == "down").astype(int)

filtered[["content_id", "trend_direction", "is_declining", "avg_position", "impressions_90d"]].head()

,content_id,trend_direction,is_declining,avg_position,impressions_90d
0,content_304f48230142,down,1,10.6,3803
1,content_a1fb4e703a9e,down,1,20.3,15320
2,content_9aa793d4d895,down,1,36.5,12581
3,content_331d6c4de07b,stable,0,6.2,11751
4,content_d99b7a2d90ca,down,1,44.0,19140


Each row is one page's snapshot: a unique `content_id`, its trend 
category, the binary target derived from it, and two of the observable 
signals (`avg_position`, `impressions_90d`) I'd use these two as features. 

This confirms the target lines up correctly with the source column and 
the data is at the grain and I would expect one page per row, not aggregated 
across pages or split across time.

Section 5: Why ML/Analysis Beats a Fixed Rule Here

With avg_position > 20 and impressions_90d > 500 as declining". I'm assuming low position and high 
impressions naturally predict decline. But my Week 1 numbers showed the 
opposite of what I expected: pages trending down actually average a 
better position (~15.9) than pages trending up (~22.5).

This is exactly why a statistical/learned approach matters here. It is letting the data show which signals actually associate with decline and in instead of encoding an assumption that turns out to be wrong. A fixed rule also can't easily combine multiple weak signals or 
adjust for known confounders like consolidation or seasonality meanwhile classification model can weigh several signals together and be validated 
against metric including ROC AUC, Precision rather than just trust on intuition.



Section 6: Self-Check

- Names the ML task type: Yes — Section 1, binary classification.

- Names the target/proxy: Yes — Section 2, `is_declining = 
  (trend_direction == "down")`, explicitly flagged as a proxy (current-
  window bucket), not a guaranteed future outcome.

- Names the success metric: Yes — Section 3, ROC AUC as primary, 
  Precision@K as secondary, chosen because accuracy would be misleading 
  given the class imbalance found in Week 1 (54.2% down).
- **Shows the unit of analysis as a real dataframe:** Yes — Section 4, 
  loaded and filtered the starter data, confirmed one row = one page via 
  `content_id`, printed actual sample rows.
- **Explains why this is ML/analysis and not just a rule:** Yes — 
  Section 5, grounded in the Week 1 finding that the position/trend 
  relationship runs opposite to intuition, which a fixed rule would get 
  wrong.
- **Ties output to a real content action:** Carried over from Week 1 
  Section 2 — flagged pages support a reviewer's triage queue, not an 
  automated decision.

**What I'm still uncertain about:** whether `trend_direction` as a 
current-window label is strong enough to build on past Week 2, or 
whether I should shift toward a future-window target before the Week 4 
lane-confirmation deadline.